Notebook 3 - Veri Hikayesi

Bu notebookta analiz sonuçları iş bakış açısıyla değerlendirilmiştir.

Problem tanımı oluşturulmuş, analiz öncesinde hipotezler belirlenmiş ve elde edilen bulgular yorumlanmıştır. Son bölümde analiz sonuçlarına dayalı iş önerileri sunulmuştur.

Problem Tanımı:
Elektrik dağıtım şirketi, Amasya iline bağlı Hamamözü, Gümüşhacıköy ve Göynücek ilçelerindeki elektrik tüketim ve tahsilat verilerini analiz ederek müşteri davranışlarını daha iyi anlamak istemektedir.

Bu çalışma kapsamında aşağıdaki sorulara cevap aranmıştır:

- İlçeler arasında elektrik tüketimi nasıl farklılaşmaktadır?
- Hangi hesap sınıfları daha fazla elektrik tüketmektedir?
- Elektrik tüketiminde mevsimsel değişimler var mıdır?
- Tahsilat süreçlerinde hangi ödeme yöntemleri daha yaygın kullanılmaktadır?
- Veri setindeki aykırı veya negatif tüketim kayıtları veri kalitesini nasıl etkilemektedir?

Bu analiz sonucunda elde edilen bulguların müşteri yönetimi, enerji planlaması ve tahsilat süreçlerinin iyileştirilmesine katkı sağlaması amaçlanmaktadır.

Hipotezler:
Bu analiz öncesinde aşağıdaki hipotezler oluşturulmuştur:

1. Gümüşhacıköy ilçesi daha fazla müşteri sayısına sahip olduğu için ortalama elektrik tüketimi diğer ilçelerden yüksektir.
2. Mesken aboneleri kayıt sayısı bakımından en büyük müşteri grubunu oluşturmaktadır.
3. Sanayi ve kurumsal abonelerin ortalama elektrik tüketimi mesken abonelerine göre daha yüksektir.
4. Yaz aylarında elektrik tüketimi kış aylarına göre artış göstermektedir.
5. Tahsilat işlemlerinin büyük çoğunluğu banka kanalı üzerinden gerçekleştirilmektedir.

In [ ]:
import pandas as pd

In [2]:
file_path="/content/elektrik_veri_hashed.xlsx"

df_hamamozu=pd.read_excel(file_path,sheet_name="Tahakkuk")
df_gumushacikoy=pd.read_excel(file_path,sheet_name="Tahakkuk 1")
df_goynucek=pd.read_excel(file_path,sheet_name="Tahakkuk 2")

df_tahakkuk=pd.concat(
    [df_hamamozu,
     df_gumushacikoy,
     df_goynucek],
    ignore_index=True
)

Bu çalışmada Hamamözü, Gümüşhacıköy ve Göynücek ilçelerine ait elektrik tüketim verileri analiz edilmiştir.

Analizin amacı ilçeler arasındaki tüketim farklılıklarını, hesap sınıflarını ve veri kalitesini inceleyerek işletmeye yönelik öneriler geliştirmektir.

In [3]:
print("Toplam kayıt:", len(df_tahakkuk))
print("Toplam sütun:", df_tahakkuk.shape[1])
print("İlçe sayısı:", df_tahakkuk["ilce"].nunique())
print("Hesap sınıfı sayısı:", df_tahakkuk["Hesap Sınıfı"].nunique())

Toplam kayıt: 1185698
Toplam sütun: 10
İlçe sayısı: 3
Hesap sınıfı sayısı: 37


Veri seti 1 milyondan fazla kayıttan oluşmaktadır. Veriler üç ilçeyi kapsamaktadır. Toplam 37 farklı hesap sınıfı bulunmaktadır.

In [4]:
ilce_analiz = df_tahakkuk.groupby("ilce")["kwh"].agg(
    Ortalama="mean",
    Medyan="median",
    Minimum="min",
    Maksimum="max",
    Kayıt="count"
).round(2)

display(ilce_analiz)

,Ortalama,Medyan,Minimum,Maksimum,Kayıt
ilce,,,,,
GÖYNÜCEK,89.67,45.09,-4208.64,105687.69,295223
GÜMÜŞHACIKÖY,97.34,48.31,-25370.64,153575.73,765657
HAMAMÖZÜ,70.87,40.56,-1242.99,25941.60,124818


Gümüşhacıköy en fazla kayıt sayısına sahip ilçedir. Hamamözü daha düşük ortalama elektrik tüketimi göstermektedir. İlçeler arasında tüketim alışkanlıklarının farklı olduğu görülmektedir.

In [5]:
hesap = df_tahakkuk.groupby("Hesap Sınıfı")["kwh"].mean().sort_values(ascending=False)

display(hesap.head(10))

,kwh
Hesap Sınıfı,
Karayolları Genel Müdürlüğü Aydınlatma,30203.431220
Aritma Tesisleri,16594.174857
Lisansız Üreticiler,16155.254444
Sanayi,7293.799626
İçme-Kullanma Suyu (Belediye),5213.509281
Tarımsal Faaliyetler (Kooperatif),3779.290533
Lisansız Üreticiler - Resmi Daire,1195.051389
"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.",1154.739118
"Resmi Üniversite,Yük.Okul,Kurs,Yurt,Okul",883.888739


En yüksek ortalama tüketim;

- Karayolları Genel Müdürlüğü Aydınlatma
- Arıtma Tesisleri
- Lisanssız Üreticiler
- Sanayi

hesap sınıflarında görülmektedir.

Mesken aboneleri ise kayıt sayısı bakımından en büyük gruptur.

In [6]:
print("Eksik kWh:", df_tahakkuk["kwh"].isna().sum())
print("Negatif kWh:", (df_tahakkuk["kwh"] < 0).sum())

Eksik kWh: 0
Negatif kWh: 151


Eksik kWh değeri bulunmamaktadır. 151 adet negatif tüketim kaydı tespit edilmiştir. Çok yüksek tüketim değerleri nedeniyle aykırı gözlemler bulunmaktadır.

In [7]:
top10 = (
    df_tahakkuk
    .groupby("Hesap Sınıfı")["kwh"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

display(top10)

,kwh
Hesap Sınıfı,
Karayolları Genel Müdürlüğü Aydınlatma,30203.431220
Aritma Tesisleri,16594.174857
Lisansız Üreticiler,16155.254444
Sanayi,7293.799626
İçme-Kullanma Suyu (Belediye),5213.509281
Tarımsal Faaliyetler (Kooperatif),3779.290533
Lisansız Üreticiler - Resmi Daire,1195.051389
"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.",1154.739118
"Resmi Üniversite,Yük.Okul,Kurs,Yurt,Okul",883.888739


Analiz sonucunda en yüksek ortalama elektrik tüketimi: Karayolları Genel Müdürlüğü Aydınlatma, Arıtma Tesisleri, Lisanssız Üreticiler ve Sanayi hesap sınıflarında görülmüştür.

Bu aboneler toplam enerji tüketiminin önemli bir bölümünü oluşturabileceğinden düzenli olarak izlenmeli ve tüketim analizleri yapılmalıdır.

In [8]:
ilce = df_tahakkuk["ilce"].value_counts()

display(ilce)

,count
ilce,
GÜMÜŞHACIKÖY,765657
GÖYNÜCEK,295223
HAMAMÖZÜ,124818


Veri setindeki kayıtların büyük bölümü Gümüşhacıköy ilçesine aittir. Bu durum hem müşteri sayısının hem de işlem hacminin diğer ilçelere göre daha yüksek olduğunu göstermektedir.

In [9]:
mesken = (
    df_tahakkuk["Hesap Sınıfı"]
    .value_counts()
    .head(10)
)

display(mesken)

,count
Hesap Sınıfı,
Mesken,1026609
Ticari Faaliyet - Yazıhane,91298
Tarımsal Faaliyetler (Şahıs),17266
İbadethane Isıtma/Soğutma/Lojman,10173
1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,8500
Şantiye ve Geçici Aboneler,6706
"Bina Ort Kul (Asn,Hidr,Kapıcı Dai vb.)",3505
İbadethane Aydınlatma,3189
Şehit Aileleri ve Gaziler,2671


Mesken aboneleri veri setindeki en büyük müşteri grubunu oluşturmaktadır. Bu nedenle yapılacak kampanya ve enerji verimliliği çalışmaları öncelikle mesken abonelerine odaklanmalıdır.

Sonuç:

Yapılan analizler sonucunda Gümüşhacıköy ilçesinin kayıt sayısı bakımından en büyük ilçe olduğu, Mesken abonelerinin en yaygın müşteri grubunu oluşturduğu ve sanayi ile kurumsal abonelerin daha yüksek elektrik tüketimine sahip olduğu görülmüştür.

Ayrıca veri setinde eksik tüketim değeri bulunmamakla birlikte sınırlı sayıda negatif tüketim kaydı tespit edilmiştir. Analiz sonuçları enerji planlaması, müşteri yönetimi ve veri kalitesinin geliştirilmesi açısından karar vericilere yol gösterecek niteliktedir.

Hipotez Sonuçları:

- Gümüşhacıköy diğer ilçelerden daha fazla kayıt sayısına sahiptir.
- Mesken hesap sınıfı açık ara en yaygın abonelik türüdür.
- Sanayi ve kurumsal abonelerin ortalama tüketimleri mesken abonelerine göre daha yüksektir.
- Elektrik tüketiminde aylık değişimler gözlenmiştir.
- Tahsilatların büyük kısmı banka üzerinden yapılmaktadır.

İş Önerileri:

- Yüksek tüketimli sanayi ve kurumsal aboneler düzenli olarak izlenmelidir.
- Negatif tüketim kayıtları sayaç veya veri giriş hatası açısından kontrol edilmelidir.
- Tahsilat işlemlerinin yoğun olduğu şubelerde operasyonel kapasite artırılabilir.
- Geç ödeme yapan aboneler için hatırlatma ve erken uyarı sistemleri geliştirilebilir. Bu yöntemle ödeme sürelerinde düzelme gözlemlenebilir.